In [1]:
import pandas as pd
import warnings
import cudf
import cupy as cp
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

## 5 Cross Validation Sampling

In [2]:
data_prep = pd.read_csv("/kaggle/input/datasets/deskafimcode/dataset-training/data_clean_review.csv")
data_prep.head()


,clean_review
0,enjoyed first season must say think season eve...
1,know iceland small country police thing bit di...
2,except k k actor look comfortable acting fight...
3,im guessing year old white woman im target dem...
4,here truth there much movie sucked high rating...


In [3]:
data_sampling = pd.read_csv("/kaggle/input/datasets/deskafimcode/dataset-training/data_sampling_labelbagging.csv")
data_sampling.head()

,review_id,movie,rating,review_detail,label_komen
0,rw5704482,After Life (2019– ),9.0,"I enjoyed the first season, but I must say I t...",1
1,rw5704483,The Valhalla Murders (2019– ),6.0,I know Iceland is a small country and police d...,0
2,rw5704484,Special OPS (2020– ),7.0,"Except K K , no other actor looks comfortable ...",0
3,rw5704485,#BlackAF (2020– ),8.0,I'm guessing that as a 62 year old white woman...,1
4,rw5704487,The Droving (2020),2.0,Here's the truth. There's not much to this mov...,0


In [4]:
data_sampling["clean_review"] = data_prep["clean_review"]
data_sampling.head()

,review_id,movie,rating,review_detail,label_komen,clean_review
0,rw5704482,After Life (2019– ),9.0,"I enjoyed the first season, but I must say I t...",1,enjoyed first season must say think season eve...
1,rw5704483,The Valhalla Murders (2019– ),6.0,I know Iceland is a small country and police d...,0,know iceland small country police thing bit di...
2,rw5704484,Special OPS (2020– ),7.0,"Except K K , no other actor looks comfortable ...",0,except k k actor look comfortable acting fight...
3,rw5704485,#BlackAF (2020– ),8.0,I'm guessing that as a 62 year old white woman...,1,im guessing year old white woman im target dem...
4,rw5704487,The Droving (2020),2.0,Here's the truth. There's not much to this mov...,0,here truth there much movie sucked high rating...


### Imbalance per Batch

In [5]:
d_data = data_sampling.to_dict(orient="list")

In [6]:
data_5_val = [{"clean_review" : [] , "label_komen" : []},{"clean_review" : [], "label_komen" : []},{"clean_review" : [], "label_komen" : []},{"clean_review" : [], "label_komen" : []},{"clean_review" : [], "label_komen" : []}]

counter = 0

for j in data_5_val :
    for k in range(data_sampling.shape[0] // 5) :
        j["clean_review"].append(d_data["clean_review"][counter])
        j["label_komen"].append(d_data["label_komen"][counter])
        counter+=1
else :
    for k in range(data_sampling.shape[0] % 5) :
        data_5_val[k]["clean_review"].append(d_data["clean_review"][counter])
        data_5_val[k]["label_komen"].append(d_data["label_komen"][counter])
        counter+=1

In [7]:
summ = 0
for i in data_5_val :
    summ += len(i["clean_review"])
summ

295357

In [8]:
for i in range(len(data_5_val)) :
    data_5_val[i] = cudf.DataFrame(data_5_val[i])

### Balance per Batch

In [9]:
data01terpisah = {"pos" : {"clean_review" : [], "label_komen" : []}, "neg" : {"clean_review" : [], "label_komen" : []}}
summm = {"pos" : 0, "neg" : 0}
for i in range(data_sampling.shape[0]) :
    if d_data["label_komen"][i] == 0 :
        data01terpisah["neg"]["clean_review"].append(d_data["clean_review"][i])
        data01terpisah["neg"]["label_komen"].append(d_data["label_komen"][i])
        summm["neg"] += 1
    else :
        data01terpisah["pos"]["clean_review"].append(d_data["clean_review"][i])
        data01terpisah["pos"]["label_komen"].append(d_data["label_komen"][i])
        summm["pos"] += 1
        
print(summm)

{'pos': 172766, 'neg': 122591}


In [10]:
data_5_val2 = [{"clean_review" : [] , "label_komen" : []},{"clean_review" : [], "label_komen" : []},{"clean_review" : [], "label_komen" : []},{"clean_review" : [], "label_komen" : []},{"clean_review" : [], "label_komen" : []}]

counter = 0
for j in data_5_val2 :
    for k in range(summm["pos"] // 5) :
        j["clean_review"].append(data01terpisah["pos"]["clean_review"][counter])
        j["label_komen"].append(data01terpisah["pos"]["label_komen"][counter])
        counter+=1
else :
    for k in range(summm["pos"] % 5) :
        j["clean_review"].append(data01terpisah["pos"]["clean_review"][counter])
        j["label_komen"].append(data01terpisah["pos"]["label_komen"][counter])
        counter+=1
        
counter = 0
for j in data_5_val2 :
    for k in range(summm["neg"] // 5) :
        j["clean_review"].append(data01terpisah["neg"]["clean_review"][counter])
        j["label_komen"].append(data01terpisah["neg"]["label_komen"][counter])
        counter+=1
else :
    for k in range(summm["neg"] % 5) :
        j["clean_review"].append(data01terpisah["neg"]["clean_review"][counter])
        j["label_komen"].append(data01terpisah["neg"]["label_komen"][counter])
        counter+=1

In [11]:
summ = 0
for i in data_5_val2 :
    summ += len(i["clean_review"])
summ

295357

In [12]:
for i in range(len(data_5_val2)) :
    data_5_val2[i] = cudf.DataFrame(data_5_val2[i])

## LETSGOOO GRID SEARCHHHH!!!!

### Penjelasan Parameter Training

model_nb_tuned = MultinomialNB(

    alpha=0.1          # Smoothing parameter: Membantu mengatasi masalah kata yang tidak ada di data training.

                       # Catatan: cuML MultinomialNB belum mendukung parameter fit_prior seperti scikit-learn.

)

model_lr_tuned = LogisticRegression(

    C=2.0,              # Regularisasi: Nilai lebih kecil = regularisasi kuat (mencegah overfitting). 
    
                        # Nilai lebih besar = model lebih fitting ke data training.
    
    solver='qn',        # Algoritma optimasi: 'qn' (Quasi-Newton/L-BFGS) adalah default di cuML yang sangat cepat 
    
                        # memanfaatkan GPU. cuML belum menyediakan solver 'saga'.
    
    max_iter=1000,      # Jumlah iterasi maksimum: Batas iterasi optimasi. Naikkan jika model belum konvergen.
    
    linesearch_max_iter=50 # Parameter khusus cuML: Batas iterasi pencarian jalur optimasi untuk menstabilkan solver 'qn'.

)

model_svm_tuned = LinearSVC(

    C=1.0,              # Penalti regularisasi: Mengontrol keseimbangan antara margin yang lebar dan meminimalkan kesalahan klasifikasi.

    loss='squared_hinge', # Fungsi loss: Fungsi penalti standar untuk LinearSVC.

    max_iter=2000,      # Batas iterasi: Ditargetkan agar kalkulasi pada data besar selesai dengan sempurna di GPU.

    fit_intercept=True  # Menentukan apakah bias/intercept perlu dihitung (secara default True di scikit-learn).

                        # Catatan: cuML LinearSVC tidak memerlukan parameter random_state karena sifat optimisasinya di GPU.

)


model_rf_tuned = RandomForestClassifier(

    n_estimators=200,      # Jumlah pohon keputusan: Lebih banyak pohon = lebih stabil dan akurat. Di GPU prosesnya akan tetap instan.

    max_depth=20,          # Kedalaman maksimum pohon: Membatasi kompleksitas pohon agar tidak overfitting dan hemat memori GPU.

    min_samples_split=5,   # Sampel minimum untuk split: Jumlah sampel minimal di sebuah node sebelum dipecah menjadi cabang baru.

    n_bins=15,             # Parameter khusus cuML: Jumlah bin diskritisasi untuk fitur kontinu. cuML menggunakan ini 

                           # untuk mempercepat kalkulasi split di GPU secara signifikan.

    random_state=42        # Konsistensi hasil: Memastikan pembuatan pohon menghasilkan hasil yang sama setiap kali dijalankan.

                           # Catatan: Parameter n_jobs=-1 dihapus karena cuML secara otomatis menggunakan seluruh core paralel GPU.
)

### Training

In [13]:
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.combine import SMOTEENN, SMOTETomek

from cuml.naive_bayes import MultinomialNB
from cuml.linear_model import LogisticRegression
from cuml.svm import LinearSVC
from cuml.ensemble import RandomForestClassifier

from cuml.feature_extraction.text import TfidfVectorizer
import heapq as hq
from cuml.metrics import accuracy_score
import gc

class kombinasi_stack :
    paket_data = None
    tf_idf_model = None
    imbalancer = None
    base_model_1 = None
    base_model_2 = None
    base_model_3 = None
    meta_model = None
    
    def __init__(self,paket_data,tf_idf_model,imbalancer,base_model_1,base_model_2,base_model_3,meta_model):
        self.paket_data = paket_data
        self.tf_idf_model = tf_idf_model
        self.imbalancer = imbalancer
        self.base_model_1 = base_model_1
        self.base_model_2 = base_model_2
        self.base_model_3 = base_model_3
        self.meta_model = meta_model

In [14]:
# data_5_batch = {"Imbalance" : data_5_val, "Balance" : data_5_val2}

# list_imbalancer = {
#     "Tanpa Imbalancer" : None,
#     "Smote" : SMOTE(random_state=42),
#     "Random Under Sampler" : RandomUnderSampler(random_state=42)
# }

# tuning_tfidf = {"ngram_range": [(1, 1), (1, 2)],"max_features": [10000],"max_df": [0.9],"min_df": [0.01], "binary": [False]}
# tuning_nb = {"alpha" : [0.01, 0.1, 0.5, 1.0, 5.0, 10.0]}
# tuning_svm = {"C": [0.1, 1, 10], "loss" : ["squared_hinge","hinge"], "max_iter" : [1000, 2000], "fit_intercept" : [True, False]}
# tuning_rf = {"n_estimators" : [100, 200], "max_depth" : [12, 16], "min_samples_leaf" : [3,5], "n_bins" : [15, 32, 64]}
# tuning_lr = {"C" : [0.01, 0.1], "penalty" : ["l1", "l2"], "max_iter" : [100, 500], "linesearch_max_iter" : [50, 100]}

# qiu_primer = []
# counter = 0
# z_counter = 0
# total_gridsearch = len(data_5_batch) * 5 * len(list_imbalancer) * len(tuning_tfidf["ngram_range"]) * len(tuning_tfidf["max_features"]) * len(tuning_tfidf["max_df"]) * len(tuning_tfidf["min_df"]) * len(tuning_tfidf["binary"])
# for h in data_5_batch:                                                                                          # 2
#     paket_data = data_5_batch[h]
#     # for i in range(len(paket_data)):                                                                            # 5
#         train = cudf.DataFrame({"clean_review" : [], "label_komen" : []})
#         test = cudf.DataFrame({"clean_review" : [], "label_komen" : []})
#         batch_data_test = i
        
#         for j in range(len(paket_data)):
#             if j == i:
#                 test = cudf.concat([test, paket_data[j]], axis=0, ignore_index=True)
#             else:
#                 train = cudf.concat([train, paket_data[j]], axis=0, ignore_index=True)
        
#         train['clean_review'] = train['clean_review'].fillna('')
#         test['clean_review'] = test['clean_review'].fillna('')
        
#         for a in tuning_tfidf["ngram_range"] :                                                                  # 3
#             for b in tuning_tfidf["max_features"] :
#                 for c in tuning_tfidf["max_df"] :
#                     for d in tuning_tfidf["min_df"] :
#                         for e in tuning_tfidf["binary"] :
                            
#                             tfidf_vectorizer = TfidfVectorizer(ngram_range= a,max_features= b,max_df= c,min_df= d,binary= e)
#                             x_train_awal = tfidf_vectorizer.fit_transform(train['clean_review'])
#                             x_test = tfidf_vectorizer.transform(test["clean_review"])
                            
#                             y_train_awal = train["label_komen"]
#                             y_test = test["label_komen"]
                            
#                             tfidf = (tfidf_vectorizer, f"TfidfVectorizer(ngram_range = {a},max_features = {b},max_df = {c},min_df = {d},binary = {e})")
#                             x_test_input = x_test.toarray()
                            
#                             for j in list_imbalancer:                                                           # 3
#                                 if j != "Tanpa Imbalancer":
#                                     x_train_dense = x_train_awal.toarray().get()
#                                     y_train_cpu = y_train_awal.to_pandas() 

#                                     x_res, y_res = list_imbalancer[j].fit_resample(x_train_dense, y_train_cpu)
                                    
#                                     x_train_input = cp.array(x_res)
#                                     y_train_input = cudf.Series(y_res)

#                                 else :
#                                     x_train_input = x_train_awal.toarray()
#                                     y_train_input = y_train_awal

#                                 qiu_sekunder = []
#                                 for k in tuning_nb["alpha"] :
#                                     model = MultinomialNB(alpha= k)
#                                     model.fit(x_train_input, y_train_input)
#                                     y_pred = model.predict(x_test_input)
#                                     akurasi = accuracy_score(y_test, y_pred)
#                                     hq.heappush(qiu_sekunder,(-akurasi,counter,model,f"MultinomialNB(alpha = {k}), Akurasi = {akurasi}"))
#                                     counter += 1
                                
#                                 hasil = hq.heappop(qiu_sekunder)
#                                 model_nb = (hasil[2],hasil[3])
#                                 y_pred_train_nb = hasil[2].predict(x_train_input)
#                                 y_pred_test_nb = hasil[2].predict(x_test_input)
                                
#                                 qiu_sekunder = []
#                                 for k in tuning_svm["C"] :
#                                     for l in tuning_svm["loss"] :
#                                         for m in tuning_svm["max_iter"] :
#                                             for n in tuning_svm["fit_intercept"] :
#                                                 model = LinearSVC(C = k, loss = l, max_iter = m, fit_intercept = n)
#                                                 model.fit(x_train_input, y_train_input)
#                                                 y_pred = model.predict(x_test_input)
#                                                 akurasi = accuracy_score(y_test, y_pred)
#                                                 hq.heappush(qiu_sekunder,(-akurasi,counter,model,f"LinearSVC(C = {k}, loss = {l}, max_iter = {m}, fit_intercept = {n}), Akurasi = {akurasi}"))
#                                                 counter += 1
                                                
#                                 hasil = hq.heappop(qiu_sekunder)
#                                 model_svm = (hasil[2],hasil[3])
#                                 y_pred_train_svm = hasil[2].predict(x_train_input)
#                                 y_pred_test_svm = hasil[2].predict(x_test_input)
                                
#                                 qiu_sekunder = []
#                                 for k in tuning_rf["n_estimators"] :
#                                     for l in tuning_rf["max_depth"] :
#                                         for m in tuning_rf["min_samples_leaf"] :
#                                             for n in tuning_rf["n_bins"] :
#                                                 model = RandomForestClassifier(n_estimators = k, max_depth = l, min_samples_leaf = m, n_bins = n, random_state = 42)
#                                                 model.fit(x_train_input, y_train_input)
#                                                 y_pred = model.predict(x_test_input)
#                                                 akurasi = accuracy_score(y_test, y_pred)
#                                                 hq.heappush(qiu_sekunder,(-akurasi,counter,model,f"RandomForestClassifier(n_estimators = {k}, max_depth = {l}, min_samples_leaf = {m}, n_bins = {n}, random_state = 42), Akurasi = {akurasi}"))
#                                                 counter += 1
                                                
#                                 hasil = hq.heappop(qiu_sekunder)
#                                 model_rf = (hasil[2],hasil[3])
#                                 y_pred_train_rf = hasil[2].predict(x_train_input)
#                                 y_pred_test_rf = hasil[2].predict(x_test_input)
                                
#                                 x_train_meta = cudf.DataFrame({"pred_nb" : y_pred_train_nb, "pred_svm" : y_pred_train_svm, "pred_rf" : y_pred_train_rf})
#                                 x_test_meta = cudf.DataFrame({"pred_nb" : y_pred_test_nb, "pred_svm" : y_pred_test_svm, "pred_rf" : y_pred_test_rf})
                                
#                                 qiu_sekunder = []
#                                 for k in tuning_lr["C"] :
#                                     for l in tuning_lr["penalty"] :
#                                         for m in tuning_lr["max_iter"] :
#                                             for n in tuning_lr["linesearch_max_iter"] :
#                                                 model = LogisticRegression(C = k, penalty = l, max_iter = m, linesearch_max_iter = n)
#                                                 model.fit(x_train_meta, y_train_input)
#                                                 y_pred = model.predict(x_test_meta)
#                                                 akurasi = accuracy_score(y_test, y_pred)
#                                                 hq.heappush(qiu_sekunder,(-akurasi,counter,model,f"LogisticRegression(C = {k}, penalty = {l}, max_iter = {m}, linesearch_max_iter = {n}), Akurasi = {akurasi}"))
#                                                 counter += 1
                                                
#                                 hasil = hq.heappop(qiu_sekunder)
#                                 model_lr = (hasil[2],hasil[3])
#                                 hq.heappush(qiu_primer,(hasil[0],counter,kombinasi_stack(paket_data= h,tf_idf_model= tfidf, imbalancer= j, base_model_1= model_nb, base_model_2= model_svm, base_model_3= model_rf, meta_model= model_lr)))
#                                 counter += 1
#                                 z_counter += 1
#                                 print(f"{z_counter}/{total_gridsearch}")
#                                 del model_nb, model_svm, model_rf, model_lr
#                                 del x_train_meta, x_test_meta
#                                 gc.collect()
#                                 cp._default_memory_pool.free_all_blocks()

### Hasil

In [15]:
# import pickle

# list_top_20 = []

# for i in range(20) :
#     p = hq.heappop(qiu_primer)
#     list_top_20.append(p[2])
#     print(f"===== Juara {i+1} =====")
#     print(f"Paket Data   = {p[2].paket_data}")
#     print(f"tf_idf       = {p[2].tf_idf_model[1]}")
#     print(f"Imbalancer   = {p[2].imbalancer}")
#     print(f"Base Model 1 = {p[2].base_model_1[1]}")
#     print(f"Base Model 2 = {p[2].base_model_2[1]}")
#     print(f"Base Model 3 = {p[2].base_model_3[1]}")
#     print(f"Meta Model   = {p[2].meta_model[1]}")
    
#     if i == 0 :
#         nama_file = "Naive_Bayes_B1.pkl"
#         with open(nama_file, 'wb') as f:
#             pickle.dump(p[2].base_model_1[0], f)
#         nama_file = "Support_Vector_Machine_B2.pkl"
#         with open(nama_file, 'wb') as f:
#             pickle.dump(p[2].base_model_2[0], f)
#         nama_file = "Random_Forest_B3.pkl"
#         with open(nama_file, 'wb') as f:
#             pickle.dump(p[2].base_model_3[0], f)
#         nama_file = "Logistic_Linear_Meta.pkl"
#         with open(nama_file, 'wb') as f:
#             pickle.dump(p[2].meta_model[0], f)
#         nama_file = "Tf_Idf.pkl"
#         with open(nama_file, 'wb') as f:
#             pickle.dump(p[2].tf_idf_model[0], f)
    


## Grid Search Part 2

In [16]:
# data_5_batch = {"Imbalance" : data_5_val}#, "Balance" : data_5_val2}

# list_imbalancer = {
#     "Tanpa Imbalancer" : None
#     # ,"Smote" : SMOTE(random_state=42)
#     # ,"Random Under Sampler" : RandomUnderSampler(random_state=42)
# }

# tuning_tfidf = {"ngram_range": [(1,2)],"max_features": [10000],"max_df": [0.9],"min_df": [0.01], "binary": [False]}
# tuning_nb = {"alpha" : [1.0]}
# tuning_svm = {"C": [10], "loss" : ["hinge"], "max_iter" : [2000], "fit_intercept" : [True]}
# tuning_rf = {"n_estimators" : [200], "max_depth" : [16], "min_samples_leaf" : [3], "n_bins" : [15]}
# tuning_lr = {"C" : [0.01], "penalty" : ["l1"], "max_iter" : [100], "linesearch_max_iter" : [50]}

# qiu_primer = []
# counter = 0
# z_counter = 0
# total_gridsearch = len(data_5_batch) * 5 * len(list_imbalancer) * len(tuning_tfidf["ngram_range"]) * len(tuning_tfidf["max_features"]) * len(tuning_tfidf["max_df"]) * len(tuning_tfidf["min_df"]) * len(tuning_tfidf["binary"])
# for h in data_5_batch:                                                                                          # 2
#     paket_data = data_5_batch[h]
#     for i in range(len(paket_data)):                                                                            # 5
#         train = cudf.DataFrame({"clean_review" : [], "label_komen" : []})
#         test = cudf.DataFrame({"clean_review" : [], "label_komen" : []})
#         batch_data_test = i
        
#         for j in range(len(paket_data)):
#             if j == i:
#                 test = cudf.concat([test, paket_data[j]], axis=0, ignore_index=True)
#             else:
#                 train = cudf.concat([train, paket_data[j]], axis=0, ignore_index=True)
        
#         train['clean_review'] = train['clean_review'].fillna('')
#         test['clean_review'] = test['clean_review'].fillna('')
        
#         for a in tuning_tfidf["ngram_range"] :                                                                  # 3
#             for b in tuning_tfidf["max_features"] :
#                 for c in tuning_tfidf["max_df"] :
#                     for d in tuning_tfidf["min_df"] :
#                         for e in tuning_tfidf["binary"] :
                            
#                             tfidf_vectorizer = TfidfVectorizer(ngram_range= a,max_features= b,max_df= c,min_df= d,binary= e)
#                             x_train_awal = tfidf_vectorizer.fit_transform(train['clean_review'])
#                             x_test = tfidf_vectorizer.transform(test["clean_review"])
                            
#                             y_train_awal = train["label_komen"]
#                             y_test = test["label_komen"]
                            
#                             tfidf = (tfidf_vectorizer, f"TfidfVectorizer(ngram_range = {a},max_features = {b},max_df = {c},min_df = {d},binary = {e})")
#                             x_test_input = x_test.toarray()
                            
#                             for j in list_imbalancer:                                                           # 3
#                                 if j != "Tanpa Imbalancer":
#                                     x_train_dense = x_train_awal.toarray().get()
#                                     y_train_cpu = y_train_awal.to_pandas() 

#                                     x_res, y_res = list_imbalancer[j].fit_resample(x_train_dense, y_train_cpu)
                                    
#                                     x_train_input = cp.array(x_res)
#                                     y_train_input = cudf.Series(y_res)

#                                 else :
#                                     x_train_input = x_train_awal.toarray()
#                                     y_train_input = y_train_awal

#                                 best_akurasi = 0
#                                 for k in tuning_nb["alpha"] :
#                                     model = MultinomialNB(alpha= k)
#                                     model.fit(x_train_input, y_train_input)
#                                     y_pred = model.predict(x_test_input)
#                                     akurasi = accuracy_score(y_test, y_pred)
#                                     if akurasi > best_akurasi:
#                                         best_akurasi = akurasi
#                                         model_nb = (model,f"MultinomialNB(alpha = {k}), Akurasi = {akurasi}")
#                                     else:
#                                         del model
#                                         cp._default_memory_pool.free_all_blocks()
#                                 y_pred_train_nb = model_nb[0].predict(x_train_input)
#                                 y_pred_test_nb = model_nb[0].predict(x_test_input)
                                
#                                 best_akurasi = 0
#                                 for k in tuning_svm["C"] :
#                                     for l in tuning_svm["loss"] :
#                                         for m in tuning_svm["max_iter"] :
#                                             for n in tuning_svm["fit_intercept"] :
#                                                 model = LinearSVC(C = k, loss = l, max_iter = m, fit_intercept = n)
#                                                 model.fit(x_train_input, y_train_input)
#                                                 y_pred = model.predict(x_test_input)
#                                                 akurasi = accuracy_score(y_test, y_pred)
#                                                 if akurasi > best_akurasi:
#                                                     best_akurasi = akurasi
#                                                     model_svm = (model,f"LinearSVC(C = {k}, loss = {l}, max_iter = {m}, fit_intercept = {n}), Akurasi = {akurasi}")
#                                                 else:
#                                                     del model
#                                                     cp._default_memory_pool.free_all_blocks()
#                                 y_pred_train_svm = model_svm[0].predict(x_train_input)
#                                 y_pred_test_svm = model_svm[0].predict(x_test_input)
                                
#                                 best_akurasi = 0
#                                 for k in tuning_rf["n_estimators"] :
#                                     for l in tuning_rf["max_depth"] :
#                                         for m in tuning_rf["min_samples_leaf"] :
#                                             for n in tuning_rf["n_bins"] :
#                                                 model = RandomForestClassifier(n_estimators = k, max_depth = l, min_samples_leaf = m, n_bins = n, random_state = 42)
#                                                 model.fit(x_train_input, y_train_input)
#                                                 y_pred = model.predict(x_test_input)
#                                                 akurasi = accuracy_score(y_test, y_pred)
#                                                 if akurasi > best_akurasi:
#                                                     best_akurasi = akurasi
#                                                     model_rf = (model,f"RandomForestClassifier(n_estimators = {k}, max_depth = {l}, min_samples_leaf = {m}, n_bins = {n}, random_state = 42), Akurasi = {akurasi}")
#                                                 else:
#                                                     del model
#                                                     cp._default_memory_pool.free_all_blocks()
#                                 y_pred_train_rf = model_rf[0].predict(x_train_input)
#                                 y_pred_test_rf = model_rf[0].predict(x_test_input)
                                
#                                 x_train_meta = cudf.DataFrame({"pred_nb" : y_pred_train_nb, "pred_svm" : y_pred_train_svm, "pred_rf" : y_pred_train_rf})
#                                 x_test_meta = cudf.DataFrame({"pred_nb" : y_pred_test_nb, "pred_svm" : y_pred_test_svm, "pred_rf" : y_pred_test_rf})
                                
#                                 best_akurasi = 0
#                                 for k in tuning_lr["C"] :
#                                     for l in tuning_lr["penalty"] :
#                                         for m in tuning_lr["max_iter"] :
#                                             for n in tuning_lr["linesearch_max_iter"] :
#                                                 model = LogisticRegression(C = k, penalty = l, max_iter = m, linesearch_max_iter = n)
#                                                 model.fit(x_train_meta, y_train_input)
#                                                 y_pred = model.predict(x_test_meta)
#                                                 akurasi = accuracy_score(y_test, y_pred)
#                                                 if akurasi > best_akurasi:
#                                                     best_akurasi = akurasi
#                                                     model_lr = (model,f"LogisticRegression(C = {k}, penalty = {l}, max_iter = {m}, linesearch_max_iter = {n}), Akurasi = {akurasi}")
#                                                 else:
#                                                     del model
#                                                     cp._default_memory_pool.free_all_blocks()
#                                 hq.heappush(qiu_primer,(-best_akurasi,counter,kombinasi_stack(paket_data= h,tf_idf_model= tfidf, imbalancer= j, base_model_1= model_nb, base_model_2= model_svm, base_model_3= model_rf, meta_model= model_lr),i))
#                                 counter += 1
#                                 z_counter += 1
#                                 print(f"{z_counter}/{total_gridsearch}")
#                                 del model_nb, model_svm, model_rf, model_lr
#                                 del x_train_meta, x_test_meta
#                                 gc.collect()
#                                 cp._default_memory_pool.free_all_blocks()

In [17]:
data_5_batch = {"Balance" : data_5_val2}

list_imbalancer = {
    "Tanpa Imbalancer" : None
    # ,"Smote" : SMOTE(random_state=42)
    # ,"Random Under Sampler" : RandomUnderSampler(random_state=42)
}

tuning_tfidf = {"ngram_range": [(1,1),(1,2)],"max_features": [10000],"max_df": [0.9],"min_df": [0.01], "binary": [False]}
tuning_nb = {"alpha" : [0.1,1.0,10]}
tuning_svm = {"C": [1,10], "loss" : ["hinge"], "max_iter" : [2000], "fit_intercept" : [True]}
tuning_rf = {"n_estimators" : [200], "max_depth" : [16], "min_samples_leaf" : [3,5], "n_bins" : [15,32,64]}
tuning_lr = {"C" : [0.01,0.1], "penalty" : ["l1"], "max_iter" : [100], "linesearch_max_iter" : [50]}

qiu_primer = []
counter = 0
z_counter = 0
total_gridsearch = len(data_5_batch) * 5 * len(list_imbalancer) * len(tuning_tfidf["ngram_range"]) * len(tuning_tfidf["max_features"]) * len(tuning_tfidf["max_df"]) * len(tuning_tfidf["min_df"]) * len(tuning_tfidf["binary"]) * len(tuning_nb["alpha"]) * len(tuning_svm["C"]) * len(tuning_svm["loss"]) * len(tuning_svm["max_iter"]) * len(tuning_svm["fit_intercept"]) * len(tuning_rf["n_estimators"]) * len(tuning_rf["max_depth"]) * len(tuning_rf["min_samples_leaf"]) * len(tuning_rf["n_bins"]) * len(tuning_lr["C"])  * len(tuning_lr["penalty"]) * len(tuning_lr["max_iter"]) * len(tuning_lr["linesearch_max_iter"])
for h in data_5_batch:                                                                                          # 2
    paket_data = data_5_batch[h]
    for i in range(len(paket_data)):                                                                            # 5
        train = cudf.DataFrame({"clean_review" : [], "label_komen" : []})
        test = cudf.DataFrame({"clean_review" : [], "label_komen" : []})
        batch_data_test = i
        
        for j in range(len(paket_data)):
            if j == i:
                test = cudf.concat([test, paket_data[j]], axis=0, ignore_index=True)
            else:
                train = cudf.concat([train, paket_data[j]], axis=0, ignore_index=True)
        
        train['clean_review'] = train['clean_review'].fillna('')
        test['clean_review'] = test['clean_review'].fillna('')
        
        for a in tuning_tfidf["ngram_range"] :                                                                  # 3
            for b in tuning_tfidf["max_features"] :
                for c in tuning_tfidf["max_df"] :
                    for d in tuning_tfidf["min_df"] :
                        for e in tuning_tfidf["binary"] :
                            
                            tfidf_vectorizer = TfidfVectorizer(ngram_range= a,max_features= b,max_df= c,min_df= d,binary= e)
                            x_train_awal = tfidf_vectorizer.fit_transform(train['clean_review'])
                            x_test = tfidf_vectorizer.transform(test["clean_review"])
                            
                            y_train_awal = train["label_komen"]
                            y_test = test["label_komen"]
                            
                            tfidf = (tfidf_vectorizer, f"TfidfVectorizer(ngram_range = {a},max_features = {b},max_df = {c},min_df = {d},binary = {e})")
                            x_test_input = x_test.toarray()
                            
                            for j in list_imbalancer:                                                           # 3
                                if j != "Tanpa Imbalancer":
                                    x_train_dense = x_train_awal.toarray().get()
                                    y_train_cpu = y_train_awal.to_pandas() 

                                    x_res, y_res = list_imbalancer[j].fit_resample(x_train_dense, y_train_cpu)
                                    
                                    x_train_input = cp.array(x_res)
                                    y_train_input = cudf.Series(y_res)

                                else :
                                    x_train_input = x_train_awal.toarray()
                                    y_train_input = y_train_awal
                                for k in tuning_nb["alpha"] :
                                    model = MultinomialNB(alpha= k)
                                    model.fit(x_train_input, y_train_input)
                                    y_pred = model.predict(x_test_input)
                                    akurasi = accuracy_score(y_test, y_pred)
                                    model_nb = (model,f"MultinomialNB(alpha = {k}), Akurasi = {akurasi}")
                                    y_pred_train_nb = model_nb[0].predict(x_train_input)
                                    y_pred_test_nb = model_nb[0].predict(x_test_input)
                                    
                                    for k in tuning_svm["C"] :
                                        for l in tuning_svm["loss"] :
                                            for m in tuning_svm["max_iter"] :
                                                for n in tuning_svm["fit_intercept"] :
                                                    model = LinearSVC(C = k, loss = l, max_iter = m, fit_intercept = n)
                                                    model.fit(x_train_input, y_train_input)
                                                    y_pred = model.predict(x_test_input)
                                                    akurasi = accuracy_score(y_test, y_pred)
                                                    model_svm = (model,f"LinearSVC(C = {k}, loss = {l}, max_iter = {m}, fit_intercept = {n}), Akurasi = {akurasi}")
                                                    y_pred_train_svm = model_svm[0].predict(x_train_input)
                                                    y_pred_test_svm = model_svm[0].predict(x_test_input)
                                                    
                                                    for o in tuning_rf["n_estimators"] :
                                                        for p in tuning_rf["max_depth"] :
                                                            for q in tuning_rf["min_samples_leaf"] :
                                                                for r in tuning_rf["n_bins"] :
                                                                    model = RandomForestClassifier(n_estimators = o, max_depth = p, min_samples_leaf = q, n_bins = r, random_state = 42)
                                                                    model.fit(x_train_input, y_train_input)
                                                                    y_pred = model.predict(x_test_input)
                                                                    akurasi = accuracy_score(y_test, y_pred)
                                                                    model_rf = (model,f"RandomForestClassifier(n_estimators = {o}, max_depth = {p}, min_samples_leaf = {q}, n_bins = {r}, random_state = 42), Akurasi = {akurasi}")
                                                                    y_pred_train_rf = model_rf[0].predict(x_train_input)
                                                                    y_pred_test_rf = model_rf[0].predict(x_test_input)
                                
                                                                    x_train_meta = cudf.DataFrame({"pred_nb" : y_pred_train_nb, "pred_svm" : y_pred_train_svm, "pred_rf" : y_pred_train_rf})
                                                                    x_test_meta = cudf.DataFrame({"pred_nb" : y_pred_test_nb, "pred_svm" : y_pred_test_svm, "pred_rf" : y_pred_test_rf})
                                
                                                                    for s in tuning_lr["C"] :
                                                                        for t in tuning_lr["penalty"] :
                                                                            for u in tuning_lr["max_iter"] :
                                                                                for v in tuning_lr["linesearch_max_iter"] :
                                                                                    model = LogisticRegression(C = s, penalty = t, max_iter = u, linesearch_max_iter = v)
                                                                                    model.fit(x_train_meta, y_train_input)
                                                                                    y_pred = model.predict(x_test_meta)
                                                                                    akurasi = accuracy_score(y_test, y_pred)
                                                                                    model_lr = (model,f"LogisticRegression(C = {s} penalty = {t}, max_iter = {u}, linesearch_max_iter = {v}), Akurasi = {akurasi}")
                                                                                    hq.heappush(qiu_primer,(-akurasi,counter,kombinasi_stack(paket_data= h,tf_idf_model= tfidf, imbalancer= j, base_model_1= model_nb, base_model_2= model_svm, base_model_3= model_rf, meta_model= model_lr),i))
                                                                                    counter += 1
                                                                                    z_counter += 1
                                                                                    print(f"{z_counter}/{total_gridsearch}")
                                                                                    gc.collect()
                                                                                    cp._default_memory_pool.free_all_blocks()

1/720
2/720
[2026-06-24 07:46:33.167] [CUML] [warning] QWL-QN line search failed (code 3); stopping at the last valid step
3/720
4/720
5/720
6/720
[2026-06-24 07:46:46.722] [CUML] [warning] QWL-QN line search failed (code 3); stopping at the last valid step
7/720
8/720
9/720
10/720
[2026-06-24 07:47:00.288] [CUML] [warning] QWL-QN stopped, because the line search failed to advance (step delta = 0.000000)
11/720
12/720
13/720
14/720
15/720
16/720
17/720
18/720
[2026-06-24 07:47:29.647] [CUML] [warning] QWL-QN stopped, because the line search failed to advance (step delta = 0.000000)
19/720
20/720
[2026-06-24 07:47:36.594] [CUML] [warning] QWL-QN stopped, because the line search failed to advance (step delta = 0.000000)
21/720
[2026-06-24 07:47:36.827] [CUML] [warning] QWL-QN stopped, because the line search failed to advance (step delta = 0.000000)
22/720
23/720
24/720
25/720
26/720
27/720
28/720
29/720
30/720
31/720
32/720
33/720
34/720
[2026-06-24 07:48:28.777] [CUML] [warning] QWL-QN

In [18]:
list_top_20_p2 = []

for i in range(total_gridsearch) :
    p = hq.heappop(qiu_primer)
    list_top_20_p2.append(p[2])
    print(f"===== Juara {i+1} =====")
    print(f"Paket Data   = {p[2].paket_data}")
    print(f"tf_idf       = {p[2].tf_idf_model[1]}")
    print(f"Imbalancer   = {p[2].imbalancer}")
    print(f"Base Model 1 = {p[2].base_model_1[1]}")
    print(f"Base Model 2 = {p[2].base_model_2[1]}")
    print(f"Base Model 3 = {p[2].base_model_3[1]}")
    print(f"Meta Model   = {p[2].meta_model[1]}")
    print(f"Batch Test   = {p[3]}")
    


===== Juara 1 =====
Paket Data   = Balance
tf_idf       = TfidfVectorizer(ngram_range = (1, 2),max_features = 10000,max_df = 0.9,min_df = 0.01,binary = False)
Imbalancer   = Tanpa Imbalancer
Base Model 1 = MultinomialNB(alpha = 1.0), Akurasi = 0.7907094851957813
Base Model 2 = LinearSVC(C = 10, loss = hinge, max_iter = 2000, fit_intercept = True), Akurasi = 0.8244993313131656
Base Model 3 = RandomForestClassifier(n_estimators = 200, max_depth = 16, min_samples_leaf = 3, n_bins = 15, random_state = 42), Akurasi = 0.73132332278106
Meta Model   = LogisticRegression(C = 0.01 penalty = l1, max_iter = 100, linesearch_max_iter = 50), Akurasi = 0.8244485449713057
Batch Test   = 2
===== Juara 2 =====
Paket Data   = Balance
tf_idf       = TfidfVectorizer(ngram_range = (1, 2),max_features = 10000,max_df = 0.9,min_df = 0.01,binary = False)
Imbalancer   = Tanpa Imbalancer
Base Model 1 = MultinomialNB(alpha = 1.0), Akurasi = 0.7907094851957813
Base Model 2 = LinearSVC(C = 10, loss = hinge, max_iter 

In [19]:
import pickle

nama_file = "Naive_Bayes_B1.pkl"
with open(nama_file, 'wb') as f:
    pickle.dump(list_top_20_p2[0].base_model_1[0], f)
nama_file = "Support_Vector_Machine_B2.pkl"
with open(nama_file, 'wb') as f:
    pickle.dump(list_top_20_p2[0].base_model_2[0], f)
nama_file = "Random_Forest_B3.pkl"
with open(nama_file, 'wb') as f:
    pickle.dump(list_top_20_p2[0].base_model_3[0], f)
nama_file = "Logistic_Linear_Meta.pkl"
with open(nama_file, 'wb') as f:
    pickle.dump(list_top_20_p2[0].meta_model[0], f)
nama_file = "Tf_Idf.pkl"
with open(nama_file, 'wb') as f:
    pickle.dump(list_top_20_p2[0].tf_idf_model[0], f)